In [3]:
#MACHINE FAILURE PREDICTION SYSTEM

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

# 1. LOAD DATA
df = pd.read_csv("Failure.csv")

print("Shape:", df.shape)
print(df.head())

# 2. CLEANING
df = df.drop([
    'UDI',
    'Product ID',
    'TWF','HDF','PWF','OSF','RNF'   #Removing because of leakage
], axis=1)

df = pd.get_dummies(df, columns=['Type'], drop_first=True)

print("\nAfter cleaning:", df.shape)

# 3. FEATURES & TARGET
X = df.drop('Machine failure', axis=1)
y = df['Machine failure']

# 4. TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 5. SCALING
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. MODEL 1 - LOGISTIC REGRESSION
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=2000
)

lr.fit(X_train, y_train)

# 7. MODEL 2 - KNN
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)

# 8. EVALUATION
def evaluate(model, X_test, y_test, name):
    pred = model.predict(X_test)
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))

evaluate(lr, X_test, y_test, "Logistic Regression")
evaluate(knn, X_test, y_test, "KNN")

# 9. FAILURE PROBABILITY OUTPUT
probs = lr.predict_proba(X_test)[:, 1]

print("\nSample Failure Probabilities (%):")
print((probs[:10] * 100).round(2))

# 10. SAVE MODEL
pickle.dump(lr, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("\nModel saved successfully!")

Shape: (10000, 14)
   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0  